# Quantize Whisper with Optimum (ONNX)

This notebook demonstrates how to export OpenAI's Whisper model to ONNX format and apply quantization using Hugging Face's Optimum library. ONNX quantization reduces model size and improves inference speed while maintaining accuracy.

We'll cover:
- Exporting Whisper to ONNX format
- Applying dynamic quantization
- Comparing model sizes
- Benchmarking inference performance

In [ ]:
!pip install optimum[onnxruntime] transformers datasets

## Export Whisper to ONNX

In [ ]:
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
from transformers import AutoProcessor

# Export Whisper to ONNX format
model_id = "openai/whisper-small"
save_dir = "./whisper-small-onnx"

print(f"Exporting {model_id} to ONNX...")
model = ORTModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    export=True
)

# Save the exported model
model.save_pretrained(save_dir)

# Also save the processor
processor = AutoProcessor.from_pretrained(model_id)
processor.save_pretrained(save_dir)

print(f"Model exported and saved to {save_dir}")

## Apply Dynamic Quantization

In [ ]:
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig
import os

quantized_dir = "./whisper-small-onnx-quantized"
os.makedirs(quantized_dir, exist_ok=True)

# Quantize the encoder
encoder_quantizer = ORTQuantizer.from_pretrained(save_dir, file_name="encoder_model.onnx")

# Use avx2 for broad CPU compatibility (also available: avx512, avx512_vnni, arm64)
qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)

print("Quantizing encoder...")
encoder_quantizer.quantize(
    save_dir=quantized_dir,
    quantization_config=qconfig,
    file_suffix=""
)

# Quantize the decoder
decoder_quantizer = ORTQuantizer.from_pretrained(save_dir, file_name="decoder_model.onnx")
print("Quantizing decoder...")
decoder_quantizer.quantize(
    save_dir=quantized_dir,
    quantization_config=qconfig,
    file_suffix=""
)

# Quantize the decoder with past key values
decoder_wp_quantizer = ORTQuantizer.from_pretrained(save_dir, file_name="decoder_with_past_model.onnx")
print("Quantizing decoder with past...")
decoder_wp_quantizer.quantize(
    save_dir=quantized_dir,
    quantization_config=qconfig,
    file_suffix=""
)

# Copy processor and config files
import shutil
for file in ["config.json", "preprocessor_config.json", "tokenizer_config.json", 
             "vocab.json", "merges.txt", "normalizer.json", "added_tokens.json", 
             "special_tokens_map.json", "generation_config.json"]:
    src = os.path.join(save_dir, file)
    if os.path.exists(src):
        shutil.copy(src, quantized_dir)

print(f"Quantized model saved to {quantized_dir}")

## Compare Model Sizes

In [ ]:
import os

def get_dir_size(path):
    """Calculate total size of all ONNX files in directory"""
    total = 0
    for file in os.listdir(path):
        if file.endswith('.onnx'):
            total += os.path.getsize(os.path.join(path, file))
    return total

original_size = get_dir_size(save_dir)
quantized_size = get_dir_size(quantized_dir)

print("Model Size Comparison:")
print(f"Original ONNX model: {original_size / 1024 / 1024:.2f} MB")
print(f"Quantized ONNX model: {quantized_size / 1024 / 1024:.2f} MB")
print(f"Size reduction: {(1 - quantized_size/original_size) * 100:.2f}%")
print(f"Compression ratio: {original_size/quantized_size:.2f}x")

## Run Inference with Quantized Model

In [ ]:
from transformers import AutoProcessor
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
from datasets import load_dataset
import torch

# Load a sample audio
print("Loading sample audio...")
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
sample = ds[0]["audio"]

# Load processor
processor = AutoProcessor.from_pretrained(save_dir)

# Prepare input
inputs = processor(sample["array"], sampling_rate=sample["sampling_rate"], return_tensors="pt")

# Load original ONNX model
print("\nRunning inference with original ONNX model...")
original_model = ORTModelForSpeechSeq2Seq.from_pretrained(save_dir)
original_outputs = original_model.generate(**inputs)
original_transcription = processor.batch_decode(original_outputs, skip_special_tokens=True)[0]
print(f"Original: {original_transcription}")

# Load quantized model
print("\nRunning inference with quantized model...")
quantized_model = ORTModelForSpeechSeq2Seq.from_pretrained(quantized_dir)
quantized_outputs = quantized_model.generate(**inputs)
quantized_transcription = processor.batch_decode(quantized_outputs, skip_special_tokens=True)[0]
print(f"Quantized: {quantized_transcription}")

# Compare outputs
print("\nOutput comparison:")
print(f"Outputs match: {original_transcription == quantized_transcription}")
if original_transcription != quantized_transcription:
    print("Note: Minor differences in transcription may occur due to quantization")

## Benchmark Latency

In [ ]:
import time
import numpy as np

num_runs = 10
print(f"Benchmarking over {num_runs} runs...\n")

# Benchmark original model
original_times = []
for i in range(num_runs):
    start = time.time()
    _ = original_model.generate(**inputs)
    end = time.time()
    original_times.append(end - start)
    print(f"Original run {i+1}: {original_times[-1]:.3f}s")

print()

# Benchmark quantized model
quantized_times = []
for i in range(num_runs):
    start = time.time()
    _ = quantized_model.generate(**inputs)
    end = time.time()
    quantized_times.append(end - start)
    print(f"Quantized run {i+1}: {quantized_times[-1]:.3f}s")

# Calculate statistics
original_mean = np.mean(original_times)
original_std = np.std(original_times)
quantized_mean = np.mean(quantized_times)
quantized_std = np.std(quantized_times)

print("\n" + "="*50)
print("Benchmark Results:")
print("="*50)
print(f"Original model:  {original_mean:.3f}s ± {original_std:.3f}s")
print(f"Quantized model: {quantized_mean:.3f}s ± {quantized_std:.3f}s")
print(f"\nSpeedup: {original_mean/quantized_mean:.2f}x")
print(f"Latency reduction: {(1 - quantized_mean/original_mean) * 100:.2f}%")